
1. Retrieve-only contextual compression (no generation)
2. Conversational RAG with memory, **without** an agent
3. That same RAG pipeline wrapped as a **tool inside an agent**
4. `MultiQueryRetriever`-based retrieval + generation, with sources


## Migration cheat-sheet

| Legacy piece | v1 (`langchain==1.3.11`) |
|---|---|
| `langchain.chat_models.ChatOpenAI` / `langchain.embeddings.OpenAIEmbeddings` | `langchain_openai.ChatOpenAI` / `langchain_openai.OpenAIEmbeddings` |
| `langchain.vectorstores.FAISS` | `langchain_community.vectorstores.FAISS` |
| `langchain.text_splitter.CharacterTextSplitter` | `langchain_text_splitters.CharacterTextSplitter` |
| `langchain.schema.Document` | `langchain_core.documents.Document` |
| `langchain.retrievers.ContextualCompressionRetriever` / `.document_compressors.LLMChainExtractor` | `langchain_classic.retrievers.ContextualCompressionRetriever` / `langchain_classic.retrievers.document_compressors.LLMChainExtractor` *(verified import — `pip install langchain-classic`)* |
| `langchain.retrievers.multi_query.MultiQueryRetriever` | `langchain_classic.retrievers.multi_query.MultiQueryRetriever` *(verified import)* |
| `retriever.get_relevant_documents(query)` | `retriever.invoke(query)` |
| `ConversationalRetrievalChain` + `ConversationBufferMemory` | Deprecated, moved to `langchain-classic`. Recommended v1 replacement: explicit retrieval + an explicit chat-history list (shown below) |
| `RetrievalQA.from_chain_type(...)` | Deprecated. Recommended v1 replacement: `retriever.invoke(query)` followed by a direct `llm.invoke(...)` call over the retrieved context |
| `initialize_agent(..., AgentType.CONVERSATIONAL_REACT_DESCRIPTION)` + `StructuredTool` | `create_agent(model, tools, checkpointer=...)` + `@tool` |

Reference: <https://docs.langchain.com/oss/python/releases/langchain-v1#create_agent>


In [1]:
!pip install -U "langchain==1.3.11" langchain-google-genai langchain-community langchain-text-splitters faiss-cpu python-dotenv langgraph

  Attempting uninstall: langchain
    Found existing installation: langchain 1.4.10
    Uninstalling langchain-1.4.10:
      Successfully uninstalled langchain-1.4.10


In [2]:

# Required imports

from langchain_ollama import (
    ChatOllama,
    OllamaEmbeddings
)

from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.documents import Document


# 1. No API Key required for Ollama!
# Make sure Ollama is installed and running.


# 2. Load the text file

with open("sample.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()


# 3. Split text into chunks

splitter = CharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = splitter.split_text(raw_text)

documents = [
    Document(page_content=chunk)
    for chunk in chunks
]


# 4. Create Ollama Embeddings

embedding = OllamaEmbeddings(
    model="nomic-embed-text"
)


# 5. Create FAISS Vector Store

vectorstore = FAISS.from_documents(
    documents,
    embedding=embedding
)


# 6. Setup Ollama LLM

llm = ChatOllama(
    model="llama3.2",
    temperature=0
)


print("Ollama LLM and FAISS Vector Store initialized successfully!")


Ollama LLM and FAISS Vector Store initialized successfully!


## 1) Retrieve-only contextual compression

`ContextualCompressionRetriever` and `LLMChainExtractor` still work exactly as before — they now live in **`langchain-classic`** rather than `langchain`. The only functional change is swapping the deprecated `.get_relevant_documents()` for `.invoke()`.

In [4]:
# ==========================================
# 1. REQUIRED IMPORTS
# ==========================================

from langchain_ollama import ChatOllama

from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor


# ==========================================
# 2. INITIALIZE OLLAMA LLM
# ==========================================

llm = ChatOllama(
    model="llama3.2",
    temperature=0
)


# ==========================================
# 3. CREATE LLM COMPRESSOR
# ==========================================

compressor = LLMChainExtractor.from_llm(llm)


# ==========================================
# 4. CREATE COMPRESSION RETRIEVER
# ==========================================

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=vectorstore.as_retriever()
)


# ==========================================
# 5. RETRIEVE AND COMPRESS DOCUMENTS
# ==========================================

print("🔹 Contextual Compression Results:")

results = compression_retriever.invoke(
    "Who created LangChain?"
)


# ==========================================
# 6. DISPLAY COMPRESSED DOCUMENTS
# ==========================================

for doc in results:
    print("-", doc.page_content)

🔹 Contextual Compression Results:
- Harrison Chase


## 2) Conversational RAG with memory — without agent mode

`ConversationalRetrievalChain` and `ConversationBufferMemory` are deprecated (moved to `langchain-classic`). Since this scenario is specifically about *not* using an agent framework, the v1-native way to do it is the same thing LangChain's own migration guide recommends: skip the chain class entirely and write the retrieval + history-aware prompt explicitly. It's a few more lines, but there's no chain black-box and no memory object to wire up.

In [7]:
import shutil
import os

# Delete old Gemini FAISS database
if os.path.exists("faiss_index"):
    shutil.rmtree("faiss_index")

print("Old FAISS database deleted!")

Old FAISS database deleted!


In [9]:
if os.path.exists("faiss_index/source.sha256"):
    os.remove("faiss_index/source.sha256")

In [10]:
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

# Ollama embedding model
embedding = OllamaEmbeddings(
    model="nomic-embed-text"
)

# Create fresh FAISS database
vectorstore = FAISS.from_documents(
    documents,
    embedding
)

# Save new database
vectorstore.save_local("faiss_index")

print("New Ollama FAISS database created!")

New Ollama FAISS database created!


In [12]:
# ==========================================
# 1. IMPORTS
# ==========================================

from langchain_ollama import (
    ChatOllama,
    OllamaEmbeddings
)

from langchain_community.vectorstores import FAISS

from langchain_classic.retrievers import ContextualCompressionRetriever

from langchain_classic.retrievers.document_compressors import LLMChainExtractor


# ==========================================
# 2. INITIALIZE OLLAMA EMBEDDINGS
# ==========================================

embedding = OllamaEmbeddings(
    model="nomic-embed-text"
)


# ==========================================
# 3. INITIALIZE FAISS VECTOR STORE
# ==========================================

# Assumes vectorstore was already created
# using the same Ollama embedding model.

# If not already created, load your saved index:
#
# vectorstore = FAISS.load_local(
#     "faiss_index",
#     embedding,
#     allow_dangerous_deserialization=True
# )


# ==========================================
# 4. INITIALIZE OLLAMA LLM
# ==========================================

llm = ChatOllama(
    model="llama3.2",
    temperature=0
)


# ==========================================
# 5. CREATE LLM COMPRESSOR
# ==========================================

compressor = LLMChainExtractor.from_llm(
    llm=llm
)


# ==========================================
# 6. CREATE COMPRESSION RETRIEVER
# ==========================================

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=vectorstore.as_retriever(
        search_kwargs={"k": 2}
    )
)


# ==========================================
# 7. CONVERSATION MEMORY
# ==========================================

chat_history: list[tuple[str, str]] = []


# ==========================================
# 8. CONVERSATIONAL RAG FUNCTION
# ==========================================

def rag_chain_fn(question: str) -> str:

    # Retrieve relevant documents
    docs = compression_retriever.invoke(question)

    # Convert documents into context
    context = "\n".join(
        doc.page_content for doc in docs
    )

    # Convert chat history into text
    history_text = "\n".join(
        f"Q: {q}\nA: {a}"
        for q, a in chat_history
    )

    # Create prompt
    prompt = (
        "You are a conversational document-based assistant.\n"
        "Answer using the conversation history and "
        "provided context.\n"
        "If the answer is not available in the context, "
        "say: Information not found in the document.\n\n"

        f"Conversation history:\n{history_text}\n\n"

        f"Context:\n{context}\n\n"

        f"Question: {question}\n\n"

        "Answer:"
    )

    # Generate answer using Ollama
    answer = llm.invoke(prompt).content

    # Save question and answer in memory
    chat_history.append((question, answer))

    return answer


# ==========================================
# 9. TEST CONVERSATIONAL RAG
# ==========================================

print("🔹 Conversational RAG with Ollama:")

print("\nQuestion 1: What is LangChain?")

print(rag_chain_fn("What is LangChain?"))


print("\nQuestion 2: Who created it?")

print(rag_chain_fn("Who created it?"))

🔹 Conversational RAG with Ollama:

Question 1: What is LangChain?
LangChain is a framework for building applications with LLMs (Large Language Models).

Question 2: Who created it?
LangChain was created by Harrison Chase.


## 3) Integration into an agent (with a tool)

The original code wrapped the chain in a `StructuredTool` and used `initialize_agent(..., AgentType.CONVERSATIONAL_REACT_DESCRIPTION)`, manually re-injecting history via `"chat_history": [memory.chat_memory.messages]` — which actually has a bug: it wraps the whole message list inside another list, one level too deep.

`create_agent` needs none of that: give it a `checkpointer` and a `thread_id`, and it tracks conversation state itself, so there's no chat-history plumbing left to get wrong.

In [13]:
!pip install -U langchain-ollama langchain langgraph

  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.11
    Uninstalling langchain-1.3.11:
      Successfully uninstalled langchain-1.3.11


In [15]:
# ==========================================
# 1. IMPORTS
# ==========================================

from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver


# ==========================================
# 2. INITIALIZE OLLAMA LLM
# ==========================================

from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2",
    temperature=0
)


# ==========================================
# 3. CREATE RAG TOOL
# ==========================================

@tool
def RAG_Tool(question: str) -> str:
    """Answer LangChain-related questions using document context."""

    # Retrieve relevant documents
    docs = compression_retriever.invoke(question)

    # Convert documents into context
    context = "\n".join(
        doc.page_content for doc in docs
    )

    # Generate answer using Ollama
    answer = llm.invoke(
        f"""
        Answer the question using only the context below.

        If the answer is not available in the context,
        say: Information not found in the document.

        Context:
        {context}

        Question:
        {question}

        Answer:
        """
    )

    return answer.content


# ==========================================
# 4. CREATE MEMORY CHECKPOINTER
# ==========================================

checkpointer = InMemorySaver()

thread_config = {
    "configurable": {
        "thread_id": "compression-demo-1"
    }
}


# ==========================================
# 5. CREATE AGENT
# ==========================================

agent = create_agent(
    model=llm,
    tools=[RAG_Tool],
    checkpointer=checkpointer,
)


# ==========================================
# 6. TEST AGENT CONVERSATION
# ==========================================

print("\n🔹 Agent Conversation:")


# Question 1
res1 = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is LangChain?"
            }
        ]
    },
    thread_config
)

print("\nAnswer 1:")
print(res1["messages"][-1].content)


# Question 2
res2 = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Who created it?"
            }
        ]
    },
    thread_config
)

print("\nAnswer 2:")
print(res2["messages"][-1].content)


🔹 Agent Conversation:

Answer 1:
It seems like I couldn't find any information on how to use LangChain in the provided document. However, I can suggest some general steps to get started with LangChain:

1. Install the required libraries and dependencies.
2. Choose a specific tool or agent that you want to use with LangChain.
3. Follow the documentation and examples provided for that tool or agent.
4. Use the LangChain API to build and integrate your application.

For more detailed instructions, I recommend checking out the official LangChain documentation or seeking help from the LangChain community.

Answer 2:
I couldn't find any information on the creator of LangChain in the provided document. However, I can suggest some possible sources where you may be able to find this information:

* The official LangChain website or documentation
* GitHub repository for LangChain
* Research papers or articles related to LangChain
* Online communities or forums where LangChain is discussed

If y

## 4) MultiQueryRetriever

`MultiQueryRetriever` is a retrieval-only utility (still verified-importable from `langchain_classic.retrievers.multi_query`, same `.from_llm(retriever=..., llm=...)` signature as before). `RetrievalQA` — the generation chain the original notebook used on top of it — is deprecated, so generation here is done the same explicit way as the rest of this notebook: retrieve, then call the LLM directly over the retrieved context.

In [24]:

# ==========================================
# 1. IMPORT REQUIRED LIBRARIES
# ==========================================

from langchain_ollama import ChatOllama
from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver


# ==========================================
# 2. INITIALIZE OLLAMA LLM
# ==========================================

llm = ChatOllama(
    model="llama3.2:latest",
    temperature=0,
    base_url="http://localhost:11434"
)


# ==========================================
# 3. CREATE RAG TOOL
# ==========================================

@tool
def RAG_Tool(question: str) -> str:
    """Answer LangChain-related questions using retrieved document context."""

    # Retrieve documents using Contextual Compression
    docs = compression_retriever.invoke(question)

    if not docs:
        return "No relevant information found in the documents."

    # Extract context
    context = "\n\n".join(
        doc.page_content for doc in docs
    )

    # Generate answer
    prompt = f"""
    Answer the question using only the context below.

    If the answer is not available in the context,
    say that the information is not available.

    Context:
    {context}

    Question:
    {question}
    """

    answer = llm.invoke(prompt)

    return answer.content


# ==========================================
# 4. INITIALIZE CHECKPOINTER MEMORY
# ==========================================

checkpointer = InMemorySaver()

thread_config = {
    "configurable": {
        "thread_id": "compression-demo-1"
    }
}


# ==========================================
# 5. CREATE AGENT
# ==========================================

agent = create_agent(
    model=llm,
    tools=[RAG_Tool],
    checkpointer=checkpointer
)


# ==========================================
# 6. AGENT CONVERSATION - FIRST QUESTION
# ==========================================

print("\n========== AGENT CONVERSATION ==========")

res1 = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is LangChain?"
            }
        ]
    },
    config=thread_config
)

print("\nUser: What is LangChain?")
print("\nAgent:", res1["messages"][-1].content)


# ==========================================
# 7. AGENT CONVERSATION - FOLLOW-UP
# ==========================================

res2 = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Who created it?"
            }
        ]
    },
    config=thread_config
)

print("\nUser: Who created it?")
print("\nAgent:", res2["messages"][-1].content)



========== AGENT CONVERSATION ==========

User: What is LangChain?

Agent: LangChain is a framework for building applications with LLMs. It provides a set of tools and libraries that make it easy to integrate LLMs into existing applications, and to build new applications that leverage the power of LLMs.

User: Who created it?

Agent: The information is not available.


In [26]:

# ============================================================
# 1. INSTALL REQUIRED LIBRARIES
# ============================================================

# Run this separately if packages are not installed:
# !pip install -U langchain langchain-community langchain-ollama langchain-classic langchain-text-splitters langgraph faiss-cpu


# ============================================================
# 2. IMPORT REQUIRED LIBRARIES
# ============================================================

from langchain_ollama import ChatOllama, OllamaEmbeddings

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import FAISS

from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

from langchain.agents import create_agent
from langchain.tools import tool

from langgraph.checkpoint.memory import InMemorySaver


# ============================================================
# 3. INITIALIZE OLLAMA LLM
# ============================================================

llm = ChatOllama(
    model="llama3.2:latest",
    temperature=0,
    base_url="http://localhost:11434"
)

print("Ollama LLM initialized successfully!")


# ============================================================
# 4. INITIALIZE OLLAMA EMBEDDINGS
# ============================================================

embeddings = OllamaEmbeddings(
    model="nomic-embed-text:latest",
    base_url="http://localhost:11434"
)

print("Ollama Embeddings initialized successfully!")


# ============================================================
# 5. LOAD DOCUMENTS
# ============================================================

documents = [

    Document(
        page_content="""
        LangChain is a framework for building applications with LLMs.

        LangChain was created by Harrison Chase.

        LangChain supports RAG, agents, memory, tools, and more.

        It is commonly used in chatbots, document Q&A,
        and AI workflows.
        """
    )

]


# ============================================================
# 6. TEXT SPLITTING
# ============================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

splits = text_splitter.split_documents(documents)

print("Total document chunks:", len(splits))


# ============================================================
# 7. CREATE FAISS VECTOR DATABASE
# ============================================================

vectorstore = FAISS.from_documents(
    documents=splits,
    embedding=embeddings
)

print("FAISS Vector Database created successfully!")


# ============================================================
# 8. CREATE BASE RETRIEVER
# ============================================================

base_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)


# ============================================================
# 9. CONTEXTUAL COMPRESSION RETRIEVER
# ============================================================

compressor = LLMChainExtractor.from_llm(llm)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

print("Contextual Compression Retriever created!")


# ============================================================
# 10. CREATE RAG TOOL
# ============================================================

@tool
def RAG_Tool(question: str) -> str:
    """Search the knowledge base and answer factual questions using retrieved documents."""

    docs = compression_retriever.invoke(question)

    if not docs:
        return "No relevant information found in the knowledge base."

    context = "\n\n".join(
        doc.page_content for doc in docs
    )

    prompt = f"""
    You are a knowledge base assistant.

    Answer the question using ONLY the context below.

    If the answer is not available in the context,
    say that the information is not available.

    Context:
    {context}

    Question:
    {question}

    Answer:
    """

    response = llm.invoke(prompt)

    return response.content


# ============================================================
# 11. INITIALIZE CONVERSATION MEMORY
# ============================================================

checkpointer = InMemorySaver()

thread_config = {
    "configurable": {
        "thread_id": "ollama-rag-demo-1"
    }
}


# ============================================================
# 12. CREATE LANGCHAIN AGENT
# ============================================================

agent = create_agent(
    model=llm,
    tools=[RAG_Tool],
    checkpointer=checkpointer,
    system_prompt="""
    You are a RAG-based AI assistant.

    IMPORTANT RULES:

    1. Always call RAG_Tool for factual questions.
    2. Understand follow-up questions using conversation history.
    3. Convert follow-up questions into complete standalone questions.
    4. Example:
       Previous question: What is LangChain?
       Follow-up question: Who created it?
       Complete question: Who created LangChain?
    5. Pass the complete question to RAG_Tool.
    6. Answer using retrieved document information only.
    7. Never invent information.
    """
)

print("LangChain Agent created successfully!")


# ============================================================
# 13. FIRST QUESTION
# ============================================================

print("\n======================================")
print("       AGENT CONVERSATION")
print("======================================")

res1 = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is LangChain?"
            }
        ]
    },
    config=thread_config
)

print("\nUser: What is LangChain?")
print("\nAgent:", res1["messages"][-1].content)


# ============================================================
# 14. FOLLOW-UP QUESTION
# ============================================================

res2 = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Who created it?"
            }
        ]
    },
    config=thread_config
)

print("\nUser: Who created it?")
print("\nAgent:", res2["messages"][-1].content)


# ============================================================
# 15. ANOTHER FOLLOW-UP QUESTION
# ============================================================

res3 = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What are its main features?"
            }
        ]
    },
    config=thread_config
)

print("\nUser: What are its main features?")
print("\nAgent:", res3["messages"][-1].content)


# ============================================================
# 16. TEST RETRIEVED DOCUMENTS
# ============================================================

print("\n======================================")
print("       RETRIEVED SOURCE DOCUMENTS")
print("======================================")

test_docs = compression_retriever.invoke(
    "Who created LangChain?"
)

for i, doc in enumerate(test_docs, 1):
    print(f"\nSource {i}:")
    print(doc.page_content)


print("\n========== EXECUTION COMPLETED ==========")


Ollama LLM initialized successfully!
Ollama Embeddings initialized successfully!
Total document chunks: 1
FAISS Vector Database created successfully!
Contextual Compression Retriever created!
LangChain Agent created successfully!

       AGENT CONVERSATION

User: What is LangChain?

Agent: LangChain is an open-source framework for building custom data pipelines and workflows. It allows developers to create modular, reusable, and scalable data processing pipelines using a variety of tools and services.

User: Who created it?

Agent: I made a mistake earlier. The correct response from RAG_Tool is that the information is not available. I should have said that instead of providing incorrect information. To answer the original question, I don't have any information on who created LangChain.

User: What are its main features?

Agent: LangChain's main features include:

1. RAG (Reasoning and Generation): A framework for building custom data pipelines and workflows.
2. Agents: A way to define 

## Summary

- `ContextualCompressionRetriever`, `LLMChainExtractor`, and `MultiQueryRetriever` are all still usable as-is — only their import path changed, to `langchain_classic`.
- `ConversationalRetrievalChain` and `RetrievalQA` are deprecated; both are replaced here with an explicit "retrieve, then call the LLM" pattern, which also removes two real bugs from the original notebook (chat history that was never actually passed to the chain, and a chat-history list nested one level too deep).
- `initialize_agent` + `AgentType` → `create_agent` + `checkpointer`, same as the rest of this series of rewrites.

For more detail, see: <https://docs.langchain.com/oss/python/releases/langchain-v1#create_agent>